In [1]:
import yaml
from src import paths
import pandas as pd
import numpy as np
import os


In [2]:
# from dotenv import load_dotenv


# from sqlalchemy import create_engine

# # Load environment variables from .env file
# load_dotenv("../private_data/.env")

# host = os.getenv("HOST")
# db = os.getenv("DB")
# port = os.getenv("PORT")
# role = os.getenv("ROLE")
# pw = os.getenv("PASSWORD")
# engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

# Barter deals

## Load data

In [3]:
analysis_name = 'BARTER_DEALS'
raw_data_path = paths.RAW_DATA_DIR / f'{analysis_name}.parquet'
df = pd.read_parquet(raw_data_path)

df = df.rename(columns = {'title': 'deal_title', 'description': 'deal_text'})


## Filter dirty rows

- Remove items containing the word 'test'

In [4]:
df.columns

Index(['created_at', 'updated_at', 'deleted_at', 'id', 'legacy_partner_id',
       'deal_title', 'deal_text', 'creators_requirement', 'hash_tags',
       'status', 'deal_value', 'go_live_at', 'live_until', 'deal_type',
       'images', 'business_tone', 'ai_enhanced', 'accepts_international',
       'accepted_countries', 'social_requirement_type_id', 'product_name',
       'schedule_type', 'schedule_model', 'legacy_id', 'tags', 'gender',
       'featured_image', 'company_id', 'partner_id',
       'applicants_applications_count', 'cancelled_applications_count',
       'company_locations', 'completed_applications_count', 'content_types',
       'deal_created_at', 'deal_deleted_at', 'deal_id', 'deal_id_str',
       'deal_tags', 'deal_updated_at', 'first_application_at',
       'last14_days_applicants_applications_count',
       'last14_days_cancelled_applications_count',
       'last14_days_completed_applications_count',
       'last14_days_pending_applications_count',
       'last14_days_

In [5]:
print(f"Number of rows before cleaning: {len(df)}")

Number of rows before cleaning: 7274


In [6]:
mask_test = df['deal_title'].str.contains('test') | df['deal_text'].str.contains('test') | df['creators_requirement'].str.contains('test')
df = df[~mask_test]
len(df)

6581

- remove deal descriptions that are too short 

In [7]:
mask_short_text = (df['deal_text'].str.len() < 10)
df = df[~mask_short_text]
len(df)

6543

## Features

- Deal language

In [8]:
df

,created_at,updated_at,deleted_at,id,legacy_partner_id,deal_title,deal_text,creators_requirement,hash_tags,status,...,live_since,main_image,min_social_media_followers,pending_applications_count,planned_applications_count,rejected_applications_count,total_company_locations,upcoming_applications_count,first_content_type,apps_after_7_days
0,2025-02-21 09:47:19.029007,2026-01-12 07:39:16.765647,NaT,019689e1-dadd-00c8-5101-e0d07384f6dd,1170.0,Heerlijke steak bij Coco's Outback in Amsterdam,Je bent welkom om nog een persoon mee te nemen...,- Vooraf spreken we een datum en tijd af. Zond...,,draft,...,2025-02-21 09:47:19.029007,uploads/deals/019689e1-dce8-ffff-960e-60637674...,5000,15,5,24,1,5,Food,17
1,2025-03-26 08:56:24.776549,2026-01-12 07:39:16.908385,NaT,019689e1-de33-00c8-297c-7a4241d57de4,745.0,Outdoor chalkboard and markers to celebrate go...,Celebrate good times with our rectangle chalkb...,-1 reel\n-1 static post\n-Share with us for ou...,,inactive,...,2025-03-26 08:56:24.776549,uploads/deals/019689e1-e020-ffff-62e0-4a2bab36...,2500,7,2,0,1,2,UGC,2
2,2025-02-21 12:14:54.016826,2026-01-12 07:39:16.957884,NaT,019689e1-e3b3-00c8-60a3-6ff40abe4535,1081.0,Bite & Bowling,Bite & Bowling! 🎳🍽️\n\nBij Lucky’s Bowling lan...,We zoeken spontane en gezellige dames en heren...,,draft,...,2025-02-21 12:14:54.016826,uploads/deals/019689e1-e491-ffff-67c5-3bbfee3e...,5000,0,1,0,1,1,Activities,2
3,2025-03-26 10:58:10.050719,2026-01-12 07:39:17.032326,NaT,019689e1-e796-00c8-dfeb-bc61800dd2d8,1156.0,Deze fles moet jij hebben!!,"Meta barst van de standaard productreviews, ma...",•\tHet filmpje is minimaal 15 seconden en maxi...,,live,...,2025-07-11 09:08:59.840326,uploads/deals/019689e1-e877-ffff-932f-add29f52...,5000,1,3,31,1,3,Activities,3
4,2025-03-26 11:14:23.339791,2026-01-12 07:39:17.071050,NaT,019689e1-e94a-00c8-d860-fccce5580ced,1156.0,Review onze 750ml Saywhat Bottle,Onze 750ml Saywhat Bottle is de perfecte fles ...,Requirements:\n•\tHet filmpje is minimaal 15 s...,,live,...,2025-07-10 10:39:33.977533,uploads/deals/019689e1-ea6d-ffff-23a3-aab8de11...,5000,0,2,13,1,2,Activities,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7268,2026-01-25 16:22:15.938891,2026-01-31 23:03:59.373750,NaT,019bf5f6-9282-00c8-a321-d43e3e00e578,NaN,Aantrekkelijke Aanbieding voor UGC Creators🔥,<p>Je&nbsp;ontvangt&nbsp;volledige&nbsp;premiu...,<p>Wat&nbsp;we&nbsp;zoeken</p><ul><li>Creators...,"#YourMotio,#persoonlijkeontwikkeling,#mindset,...",finished,...,2026-01-25 16:22:16.036170,uploads/deals/019bf5f5-3193-ffff-310f-bbfe906a...,1500,0,3,0,0,3,Experiences,3
7269,2026-01-25 16:31:50.902867,2026-01-31 23:04:00.377427,NaT,019bf5ff-5876-00c8-35a6-e5a7dcb5369f,NaN,Maandelijks inkomen + gratis toegang tot YourM...,<p>Samenwerken&nbsp;met&nbsp;YourMotio</p><p>Y...,<p>Wat&nbsp;we&nbsp;van&nbsp;jou&nbsp;vragen</...,"#YourMotio,#persoonlijkeontwikkeling,#mindset,...",finished,...,2026-01-25 16:31:51.000818,uploads/deals/019bf5fe-e5e0-ffff-d7be-312bb2cd...,1500,0,1,0,0,1,Experiences,1
7270,2026-01-16 15:47:39.999471,2026-02-05 04:00:02.372342,NaT,019bc77d-a95f-00c8-e8c0-6cbe4e05db15,NaN,Design Wanduhren mit Charakter,<p>Von&nbsp;<strong>Pattern&nbsp;Clock</strong...,"<p>Hallo,</p><p>ich&nbsp;baue&nbsp;aktuell&nbs...","#Design,#Interior,#ModernLiving,#Wanduhr",draft,...,2026-01-16 15:47:40.110228,uploads/deals/019bc787-f22c-ffff-3fee-1b7e90ff...,1500,6,0,0,0,0,UGC,5
7272,2025-09-15 09:16:48.051838,2026-02-05 04:00:05.652332,NaT,01994ca9-9c33-00c8-4d1d-dc713a3d0713,2130.0,Erhalten Sie 75 € Shop-Guthaben! 💪,"Wir von Bandagenspezialist.de glauben, dass je...",Im Austausch für den Gutschein bitten wir Sie ...,,draft,...,2025-09-15 09:16:48.176478,uploads/deals/01994ca9-487c-ffff-aec6-a24fb0dd...,5000,7,0,0,0,0,Activities,2


In [9]:
from langdetect import detect, detect_langs

deal_langs = []
for i, deal in enumerate(df['deal_text']):
    try:
        lang = detect(deal)
        deal_langs.append(lang)
    except Exception as e:
        print(e)
        deal_langs.append(np.nan)

df['deal_language'] = deal_langs

# Afrikaans is actually Dutch
df.loc[df.deal_language == 'af','deal_language'] = 'nl'

# Filter languages that are not Dutch, English or German (they are gibberish)
allowed_languages = ['nl', 'en', 'de']
df = df[df['deal_language'].isin(allowed_languages)]
len(df)

No features in text.
No features in text.
No features in text.
No features in text.
No features in text.
No features in text.


6472

## Save

In [10]:
processed_data_path = paths.PROCESSED_DATA_DIR / f'{analysis_name}_CLEAN.parquet'
df.to_parquet(processed_data_path)